In [3]:
import pandas as pd

train_df = pd.read_csv("/content/en_fr_train.csv")
val_df = pd.read_csv("/content/en_fr_val.csv")
test_df = pd.read_csv("/content/en_fr_test.csv")

print("Train:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)

Train: (232825, 2)
Validation: (890, 2)
Test: (8597, 2)


In [4]:
print(train_df.columns)
print(train_df.head())
print(train_df.isnull().sum())

Index(['en', 'fr'], dtype='object')
                                                  en  \
0  Thank you so much, Chris. And it's truly a gre...   
1  I have been blown away by this conference, and...   
2  And I say that sincerely, partly because  Put ...   
3           I flew on Air Force Two for eight years.   
4  Now I have to take off my shoes or boots to ge...   

                                                  fr  
0  Merci beaucoup, Chris. C'est vraiment un honne...  
1  J'ai été très impressionné par cette conférenc...  
2  Et je dis çà sincèrement, en autres parce que ...  
3       J'ai volé avec Air Force 2 pendant huit ans.  
4  Et maintenant, je dois enlever mes chaussures ...  
en    0
fr    0
dtype: int64


In [5]:
import re

def tokenize(sentence):
    sentence = sentence.lower()
    sentence = re.sub(r"([?.!,])", r" \1", sentence)
    return sentence.split()

In [6]:
train_df["en_len"] = train_df["en"].apply(lambda x: len(tokenize(x)))
train_df["fr_len"] = train_df["fr"].apply(lambda x: len(tokenize(x)))

print("English average length:", train_df["en_len"].mean())
print("English maximum length:", train_df["en_len"].max())

print("French average length:", train_df["fr_len"].mean())
print("French maximum length:", train_df["fr_len"].max())

English average length: 19.71223880597015
English maximum length: 118
French average length: 20.325351658971332
French maximum length: 117


In [7]:
print(train_df[["en_len", "fr_len"]].describe())

              en_len         fr_len
count  232825.000000  232825.000000
mean       19.712239      20.325352
std        13.345255      13.897086
min         1.000000       1.000000
25%        10.000000      10.000000
50%        16.000000      17.000000
75%        26.000000      26.000000
max       118.000000     117.000000


Creating vocabularies

In [8]:
from collections import Counter

PAD = 0
SOS = 1
EOS = 2
UNK = 3

def build_vocab(sentences, max_vocab_size=10000):

    counter = Counter()

    for sentence in sentences:
        counter.update(tokenize(sentence))

    vocab = {
        "<PAD>": PAD,
        "<SOS>": SOS,
        "<EOS>": EOS,
        "<UNK>": UNK
    }

    for word, count in counter.most_common(max_vocab_size - 4):
        vocab[word] = len(vocab)

    return vocab

In [9]:
en_vocab = build_vocab(train_df["en"])
fr_vocab = build_vocab(train_df["fr"])

print("English vocabulary:", len(en_vocab))
print("French vocabulary:", len(fr_vocab))

English vocabulary: 10000
French vocabulary: 10000


In [10]:
en_idx_to_word = {index: word for word, index in en_vocab.items()}
fr_idx_to_word = {index: word for word, index in fr_vocab.items()}

Tokenizer

In [11]:
MAX_LEN = 50

def sentence_to_ids(sentence, vocab):

    tokens = tokenize(sentence)

    tokens = tokens[:MAX_LEN - 2]

    ids = [SOS]

    for token in tokens:
        ids.append(vocab.get(token, UNK))

    ids.append(EOS)

    return ids

Here the MAX-2 is basically for < SOS> tokens < EOS>

In [12]:
example = train_df["en"].iloc[0]

print(example)
print(sentence_to_ids(example, en_vocab))

Thank you so much, Chris. And it's truly a great honor to have the opportunity to come to this stage twice; I'm extremely grateful.
[1, 164, 15, 19, 110, 4, 1086, 5, 7, 30, 1053, 10, 165, 2146, 8, 23, 6, 644, 8, 147, 8, 18, 717, 3, 83, 922, 3054, 5, 2]


Now we basically got the tokens but they can be of varying length right so we need to do padding to make sure that when we send them in an RNN/ LSTM in sequential order (not parrallely as in transformers), the tensors can be processed.

In [13]:
import torch
from torch.utils.data import Dataset, DataLoader

In [14]:
MAX_LEN = 50

class TranslationDataset(Dataset):

    def __init__(self, dataframe, en_vocab, fr_vocab):
        self.en_sentences = dataframe["en"].tolist()
        self.fr_sentences = dataframe["fr"].tolist()
        self.en_vocab = en_vocab
        self.fr_vocab = fr_vocab

    def __len__(self):
        return len(self.en_sentences)

    def __getitem__(self, index):

        en_ids = sentence_to_ids(
            self.en_sentences[index],
            self.en_vocab
        )

        fr_ids = sentence_to_ids(
            self.fr_sentences[index],
            self.fr_vocab
        )

        return torch.tensor(en_ids), torch.tensor(fr_ids)

In [15]:
def collate_fn(batch):

    en_batch = [item[0] for item in batch]
    fr_batch = [item[1] for item in batch]

    en_batch = torch.nn.utils.rnn.pad_sequence(
        en_batch,
        batch_first=True,
        padding_value=PAD
    )

    fr_batch = torch.nn.utils.rnn.pad_sequence(
        fr_batch,
        batch_first=True,
        padding_value=PAD
    )

    return en_batch, fr_batch

In [16]:
train_dataset = TranslationDataset(
    train_df,
    en_vocab,
    fr_vocab
)

val_dataset = TranslationDataset(
    val_df,
    en_vocab,
    fr_vocab
)

test_dataset = TranslationDataset(
    test_df,
    en_vocab,
    fr_vocab
)

In [17]:
train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True,
    collate_fn=collate_fn
)

val_loader = DataLoader(
    val_dataset,
    batch_size=64,
    shuffle=False,
    collate_fn=collate_fn
)

test_loader = DataLoader(
    test_dataset,
    batch_size=64,
    shuffle=False,
    collate_fn=collate_fn
)

Check on 1 singular batch

In [18]:
en_batch, fr_batch = next(iter(train_loader))

print("English batch:", en_batch.shape)
print("French batch:", fr_batch.shape)

English batch: torch.Size([64, 50])
French batch: torch.Size([64, 50])


Making the RNN Encoder

In [19]:
import torch.nn as nn

In [20]:
class EncoderRNN(nn.Module):

    def __init__(self, input_size, embedding_dim, hidden_dim):
        super().__init__()

        self.embedding = nn.Embedding(
            input_size,
            embedding_dim
        )

        self.rnn = nn.RNN(
            embedding_dim,
            hidden_dim,
            batch_first=True
        )

    def forward(self, x):

        embedded = self.embedding(x)

        outputs, hidden = self.rnn(embedded)

        return hidden

In this code im returning hidden as basically all the 50 tokens are processed (max 50) fron the encoder and then sent to a decoder for further processing. So this is the final hidden state as the context passed to the decoder.

RNN DECODER

In [21]:
class DecoderRNN(nn.Module):

    def __init__(self, output_size, embedding_dim, hidden_dim):
        super().__init__()

        self.embedding = nn.Embedding(
            output_size,
            embedding_dim
        )

        self.rnn = nn.RNN(
            embedding_dim,
            hidden_dim,
            batch_first=True
        )

        self.output_layer = nn.Linear(
            hidden_dim,
            output_size
        )

    def forward(self, x, hidden):

        embedded = self.embedding(x)

        output, hidden = self.rnn(
            embedded,
            hidden
        )

        prediction = self.output_layer(output)

        return prediction, hidden

The decoder is processing words 1 word at a time

How big should the internal vectors be?
So im setting -
INPUT_SIZE = 10,000 (eng vocab)
OUTPUT_SIZE = 10,000 (french vocab)
EMBEDDING_DIM = 256
HIDDEN_DIM = 512 (Context vector ka dim)

In [22]:
INPUT_SIZE = len(en_vocab)
OUTPUT_SIZE = len(fr_vocab)

EMBEDDING_DIM = 256
HIDDEN_DIM = 512

encoder = EncoderRNN(
    INPUT_SIZE,
    EMBEDDING_DIM,
    HIDDEN_DIM
)

decoder = DecoderRNN(
    OUTPUT_SIZE,
    EMBEDDING_DIM,
    HIDDEN_DIM
)

In [23]:
print("Encoder parameters:", sum(p.numel() for p in encoder.parameters()))
print("Decoder parameters:", sum(p.numel() for p in decoder.parameters()))

Encoder parameters: 2954240
Decoder parameters: 8084240


Training function

In [24]:
import torch.optim as optim

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

encoder = encoder.to(device)
decoder = decoder.to(device)

print("Using:", device)

Using: cuda


In [25]:
import torch

print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

True
Tesla T4


In [26]:
criterion = nn.CrossEntropyLoss(ignore_index=PAD)

encoder_optimizer = optim.Adam(
    encoder.parameters(),
    lr=0.001
)

decoder_optimizer = optim.Adam(
    decoder.parameters(),
    lr=0.001
)

In [28]:
def train_epoch(encoder, decoder, train_loader):

    encoder.train()
    decoder.train()

    total_loss = 0

    for en_batch, fr_batch in train_loader:

        en_batch = en_batch.to(device)
        fr_batch = fr_batch.to(device)

        encoder_optimizer.zero_grad()
        decoder_optimizer.zero_grad()

        # Encode the English sentence
        hidden = encoder(en_batch)

        # Start with <SOS>
        decoder_input = fr_batch[:, 0].unsqueeze(1)

        loss = 0

        # Generate one French word at a time
        for t in range(1, fr_batch.size(1)):

            output, hidden = decoder(
                decoder_input,
                hidden
            )

            loss += criterion(
                output.squeeze(1),
                fr_batch[:, t]
            )

            # Teacher forcing
            decoder_input = fr_batch[:, t].unsqueeze(1)

        loss.backward()

        encoder_optimizer.step()
        decoder_optimizer.step()

        total_loss += loss.item()

    return total_loss / len(train_loader)

In [29]:
import time

start = time.time()

loss = train_epoch(
    encoder,
    decoder,
    train_loader
)

print(f"Loss: {loss:.4f}")
print(f"Time: {time.time() - start:.1f} seconds")

Loss: 220.8317
Time: 302.3 seconds


This is wayyy too slow (trying teacher forcing together not
individually)


In [30]:
def train_epoch(encoder, decoder, train_loader):

    encoder.train()
    decoder.train()

    total_loss = 0

    for en_batch, fr_batch in train_loader:

        en_batch = en_batch.to(device)
        fr_batch = fr_batch.to(device)

        encoder_optimizer.zero_grad()
        decoder_optimizer.zero_grad()

        # Encode English sentence
        hidden = encoder(en_batch)

        # Give decoder the correct previous words
        decoder_input = fr_batch[:, :-1]

        # Correct words that the decoder should predict
        target = fr_batch[:, 1:]

        # Decode the whole sequence at once
        output, hidden = decoder(
            decoder_input,
            hidden
        )

        loss = criterion(
            output.reshape(-1, output.size(-1)),
            target.reshape(-1)
        )

        loss.backward()

        encoder_optimizer.step()
        decoder_optimizer.step()

        total_loss += loss.item()

    return total_loss / len(train_loader)

In [31]:
import time

start = time.time()

loss = train_epoch(
    encoder,
    decoder,
    train_loader
)

print(f"Loss: {loss:.4f}")
print(f"Time: {time.time() - start:.1f} seconds")

Loss: 4.1206
Time: 152.7 seconds


We alr kinda trained 2 epochs so training 2 more epochs now-

In [32]:
for epoch in range(2):

    start = time.time()

    loss = train_epoch(
        encoder,
        decoder,
        train_loader
    )

    print(
        f"Epoch {epoch + 3}/4 - "
        f"Loss: {loss:.4f} - "
        f"Time: {time.time() - start:.1f}s"
    )

Epoch 3/4 - Loss: 4.0296 - Time: 153.2s
Epoch 4/4 - Loss: 4.0123 - Time: 151.3s


Generate Translations

In [33]:
def translate_sentence(sentence):

    encoder.eval()
    decoder.eval()

    tokens = tokenize(sentence)
    tokens = tokens[:MAX_LEN - 2]

    ids = [SOS]

    for token in tokens:
        ids.append(en_vocab.get(token, UNK))

    ids.append(EOS)

    source = torch.tensor(ids).unsqueeze(0).to(device)

    with torch.no_grad():

        hidden = encoder(source)

        decoder_input = torch.tensor([[SOS]]).to(device)

        generated_words = []

        for _ in range(MAX_LEN):

            output, hidden = decoder(
                decoder_input,
                hidden
            )

            next_token = output.argmax(2)[:, -1].item()

            if next_token == EOS:
                break

            if next_token != PAD:
                generated_words.append(
                    fr_idx_to_word.get(next_token, "<UNK>")
                )

            decoder_input = torch.tensor(
                [[next_token]]
            ).to(device)

    return generated_words

Testing (on few examples)

In [34]:
for i in range(5):

    english = test_df["en"].iloc[i]
    actual_french = test_df["fr"].iloc[i]

    prediction = translate_sentence(english)

    print("\nEN:", english)
    print("TRUE:", actual_french)
    print("PRED:", " ".join(prediction))


EN: Several years ago here at TED, Peter Skillman  introduced a design challenge  called the marshmallow challenge.
TRUE: Il y a plusieurs années, ici à Ted, Peter Skillman a présenté une épreuve de conception appelée l'épreuve du marshmallow.
PRED: et je pense que c'est un peu plus de <UNK> .

EN: And the idea's pretty simple:  Teams of four have to build the tallest free-standing structure  out of 20 sticks of spaghetti,  one yard of tape, one yard of string  and a marshmallow.
TRUE: Et l'idée est plutôt simple. Des équipes de quatre personnes doivent bâtir la plus haute structure tenant debout avec 20 spaghettis, un mètre de ruban collant, un mètre de ficelle, et un marshmallow.
PRED: et je pense que c'est un peu plus de <UNK> .

EN: The marshmallow has to be on top.
TRUE: Le marshmallow doit être placé au sommet.
PRED: et je pense que c'est un peu plus de <UNK> .

EN: And, though it seems really simple, it's actually pretty hard  because it forces people  to collaborate very quick

In [35]:
for i in range(3):

    english = train_df["en"].iloc[i]
    actual_french = train_df["fr"].iloc[i]

    prediction = translate_sentence(english)

    print("\nEN:", english)
    print("TRUE:", actual_french)
    print("PRED:", " ".join(prediction))


EN: Thank you so much, Chris. And it's truly a great honor to have the opportunity to come to this stage twice; I'm extremely grateful.
TRUE: Merci beaucoup, Chris. C'est vraiment un honneur de pouvoir venir sur cette scène une deuxième fois. Je suis très reconnaissant.
PRED: et je pense que c'est un peu plus de <UNK> .

EN: I have been blown away by this conference, and I want to thank all of you for the many nice comments about what I had to say the other night.
TRUE: J'ai été très impressionné par cette conférence, et je tiens à vous remercier tous pour vos nombreux et sympathiques commentaires sur ce que j'ai dit l'autre soir.
PRED: et je pense que c'est un peu plus de <UNK> .

EN: And I say that sincerely, partly because  Put yourselves in my position.
TRUE: Et je dis çà sincèrement, en autres parce que --Faux sanglot-- j'en ai besoin ! --Rires-- Mettez-vous à ma place!
PRED: et je pense que c'est un peu plus de <UNK> .


(Same probelm its repeatedly giving same o/p so not a generalization probem anymore... )

In [36]:
import nltk
from nltk.translate.bleu_score import corpus_bleu

nltk.download("punkt")

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


True

BLEU SCORE

In [37]:
from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction

def translate_batch(en_batch):
    encoder.eval()
    decoder.eval()

    en_batch = en_batch.to(device)

    with torch.no_grad():
        hidden = encoder(en_batch)

        batch_size = en_batch.size(0)

        decoder_input = torch.full(
            (batch_size, 1),
            SOS,
            dtype=torch.long,
            device=device
        )

        generated = [[] for _ in range(batch_size)]
        finished = [False] * batch_size

        for _ in range(MAX_LEN):

            output, hidden = decoder(
                decoder_input,
                hidden
            )

            next_tokens = output[:, -1, :].argmax(dim=1)

            for i, token in enumerate(next_tokens.tolist()):
                if not finished[i]:

                    if token == EOS:
                        finished[i] = True

                    elif token != PAD:
                        generated[i].append(
                            fr_idx_to_word.get(token, "<UNK>")
                        )

            decoder_input = next_tokens.unsqueeze(1)

            if all(finished):
                break

    return generated

In [38]:
references = []
hypotheses = []

for en_batch, fr_batch in test_loader:

    predictions = translate_batch(en_batch)
    hypotheses.extend(predictions)

    # Get the corresponding reference sentences
    start = len(references)

    for i in range(len(predictions)):
        reference = tokenize(
            test_df["fr"].iloc[start + i]
        )
        references.append([reference])

bleu = corpus_bleu(
    references,
    hypotheses,
    smoothing_function=SmoothingFunction().method1
)

print("RNN BLEU:", bleu)

RNN BLEU: 0.0028648525392761166


(Obviously very less because of the problems stated in experiment_log but we move).

# SAME THING WITH LSTMS



In [39]:
class EncoderLSTM(nn.Module):
    def __init__(self, input_size, embedding_dim, hidden_dim):
        super().__init__()

        self.embedding = nn.Embedding(input_size, embedding_dim)
        self.lstm = nn.LSTM(
            embedding_dim,
            hidden_dim,
            batch_first=True
        )

    def forward(self, x):
        embedded = self.embedding(x)
        output, (hidden, cell) = self.lstm(embedded)

        return hidden, cell

In [40]:
class DecoderLSTM(nn.Module):
    def __init__(self, output_size, embedding_dim, hidden_dim):
        super().__init__()

        self.embedding = nn.Embedding(output_size, embedding_dim)
        self.lstm = nn.LSTM(
            embedding_dim,
            hidden_dim,
            batch_first=True
        )
        self.output_layer = nn.Linear(hidden_dim, output_size)

    def forward(self, x, hidden, cell):
        embedded = self.embedding(x)

        output, (hidden, cell) = self.lstm(
            embedded,
            (hidden, cell)
        )

        prediction = self.output_layer(output)

        return prediction, hidden, cell

In [41]:
encoder_lstm = EncoderLSTM(
    INPUT_SIZE,
    EMBEDDING_DIM,
    HIDDEN_DIM
).to(device)

decoder_lstm = DecoderLSTM(
    OUTPUT_SIZE,
    EMBEDDING_DIM,
    HIDDEN_DIM
).to(device)

In [42]:
encoder_optimizer = optim.Adam(
    encoder_lstm.parameters(),
    lr=0.001
)

decoder_optimizer = optim.Adam(
    decoder_lstm.parameters(),
    lr=0.001
)

In [43]:
def train_epoch_lstm(encoder, decoder, train_loader):
    encoder.train()
    decoder.train()

    total_loss = 0

    for en_batch, fr_batch in train_loader:

        en_batch = en_batch.to(device)
        fr_batch = fr_batch.to(device)

        encoder_optimizer.zero_grad()
        decoder_optimizer.zero_grad()

        hidden, cell = encoder(en_batch)

        decoder_input = fr_batch[:, :-1]
        target = fr_batch[:, 1:]

        output, hidden, cell = decoder(
            decoder_input,
            hidden,
            cell
        )

        loss = criterion(
            output.reshape(-1, output.size(-1)),
            target.reshape(-1)
        )

        loss.backward()

        encoder_optimizer.step()
        decoder_optimizer.step()

        total_loss += loss.item()

    return total_loss / len(train_loader)

In [44]:
def count_parameters(model):
    return sum(p.numel() for p in model.parameters())

print("Encoder parameters:", count_parameters(encoder_lstm))
print("Decoder parameters:", count_parameters(decoder_lstm))
print("Total parameters:",
      count_parameters(encoder_lstm) + count_parameters(decoder_lstm))

Encoder parameters: 4136960
Decoder parameters: 9266960
Total parameters: 13403920


In [45]:
import time

start_time = time.time()

for epoch in range(4):

    loss = train_epoch_lstm(
        encoder_lstm,
        decoder_lstm,
        train_loader
    )

    print(
        f"Epoch {epoch+1}/4, "
        f"Loss: {loss:.4f}"
    )

print("Training time:", time.time() - start_time)

Epoch 1/4, Loss: 4.1930
Epoch 2/4, Loss: 3.5002
Epoch 3/4, Loss: 3.2120
Epoch 4/4, Loss: 3.0168
Training time: 862.2084529399872


EVALUATION

In [46]:
def translate_sentence_lstm(sentence):
    encoder_lstm.eval()
    decoder_lstm.eval()

    tokens = tokenize(sentence)
    tokens = tokens[:MAX_LEN - 2]

    ids = [SOS]

    for token in tokens:
        ids.append(en_vocab.get(token, UNK))

    ids.append(EOS)

    source = torch.tensor(ids).unsqueeze(0).to(device)

    with torch.no_grad():
        hidden, cell = encoder_lstm(source)

        decoder_input = torch.tensor([[SOS]]).to(device)

        generated_words = []

        for _ in range(MAX_LEN):

            output, hidden, cell = decoder_lstm(
                decoder_input,
                hidden,
                cell
            )

            next_token = output[:, -1, :].argmax(1).item()

            if next_token == EOS:
                break

            if next_token != PAD:
                generated_words.append(
                    fr_idx_to_word.get(next_token, "<UNK>")
                )

            decoder_input = torch.tensor(
                [[next_token]]
            ).to(device)

    return generated_words

In [47]:
for i in range(5):
    sentence = test_df["en"].iloc[i]

    print("English:", sentence)
    print("Actual:", test_df["fr"].iloc[i])
    print("LSTM:", " ".join(translate_sentence_lstm(sentence)))
    print()

English: Several years ago here at TED, Peter Skillman  introduced a design challenge  called the marshmallow challenge.
Actual: Il y a plusieurs années, ici à Ted, Peter Skillman a présenté une épreuve de conception appelée l'épreuve du marshmallow.
LSTM: il y a quelques années , il y a une histoire de ted , qui est un projet <UNK> ,

English: And the idea's pretty simple:  Teams of four have to build the tallest free-standing structure  out of 20 sticks of spaghetti,  one yard of tape, one yard of string  and a marshmallow.
Actual: Et l'idée est plutôt simple. Des équipes de quatre personnes doivent bâtir la plus haute structure tenant debout avec 20 spaghettis, un mètre de ruban collant, un mètre de ficelle, et un marshmallow.
LSTM: et <UNK> <UNK> a fait une série de <UNK> de <UNK> de <UNK> <UNK> qui <UNK> beaucoup de <UNK> , et ils ont découvert que la <UNK> est un peu plus <UNK> , de plus

English: The marshmallow has to be on top.
Actual: Le marshmallow doit être placé au sommet.

In [48]:
def translate_batch_lstm(en_batch):
    encoder_lstm.eval()
    decoder_lstm.eval()

    en_batch = en_batch.to(device)

    with torch.no_grad():
        hidden, cell = encoder_lstm(en_batch)

        batch_size = en_batch.size(0)

        decoder_input = torch.full(
            (batch_size, 1),
            SOS,
            dtype=torch.long,
            device=device
        )

        generated = [[] for _ in range(batch_size)]
        finished = [False] * batch_size

        for _ in range(MAX_LEN):

            output, hidden, cell = decoder_lstm(
                decoder_input,
                hidden,
                cell
            )

            next_tokens = output[:, -1, :].argmax(dim=1)

            for i, token in enumerate(next_tokens.tolist()):

                if not finished[i]:

                    if token == EOS:
                        finished[i] = True

                    elif token != PAD:
                        generated[i].append(
                            fr_idx_to_word.get(
                                token,
                                "<UNK>"
                            )
                        )

            decoder_input = next_tokens.unsqueeze(1)

            if all(finished):
                break

    return generated

In [49]:
references = []
hypotheses = []

for en_batch, fr_batch in test_loader:

    predictions = translate_batch_lstm(en_batch)

    hypotheses.extend(predictions)

    start = len(references)

    for i in range(len(predictions)):
        reference = tokenize(
            test_df["fr"].iloc[start + i]
        )

        references.append([reference])

bleu_lstm = corpus_bleu(
    references,
    hypotheses,
    smoothing_function=SmoothingFunction().method1
)

print("LSTM BLEU:", bleu_lstm)

LSTM BLEU: 0.053256901089099994
